In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import vars

import pandas as pd, numpy as np, matplotlib.pyplot as plt
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 20)
pd.set_option('display.max_colwidth', 200)


path_repo = vars.path_repo
path_data = vars.path_data

In [ ]:
symbs = vars.symbs

import joblib,os
symbs = joblib.load(path_repo + "pkl/best_symbols.pkl") Ajustar y definir como se calcula

symbs_crypto = [x for x in symbs if "-USD" in x]
symbs_forex = [x for x in symbs if "=X" in x]
symbs_stocks = [x for x in symbs if x not in symbs_crypto and x not in symbs_forex]
symbs = list(set(symbs))
print("Total:", len(symbs))
print("Crypto :", len(symbs_crypto), f"({(len(symbs_crypto)/len(symbs))*100:.0f}%)")
print("Forex:", len(symbs_forex), f"({(len(symbs_forex)/len(symbs))*100:.0f}%)")
print("Stocks:", len(symbs_stocks), f"({(len(symbs_stocks)/len(symbs))*100:.0f}%)")

In [ ]:
# Load Symbol
import yfinance as yf
SYMBOLS = symbs 

# EMAS
EMA_SHORT       = 20
EMA_MID         = 50
EMA_LONG        = 200
# RSI
RSI_PERIOD      = 14
# MACD
MACD_FAST       = 12
MACD_SLOW       = 26
MACD_SIGNAL     = 9
# Bollinger
BB_PERIOD       = 20
BB_STD          = 2.0
# Estocastico
STOCH_K         = 9
STOCH_D         = 3
STOCH_SLOWING   = 3  
# ATR
ATR_PERIOD      = 14
# ATR - Gestión de riesgo
ATR_MULT_SL     = 1.5           # Multiplicador ATR para Stop Loss
ATR_MULT_TP     = 3.0           # Multiplicador ATR para Take Profit (ratio 2:1)
# Parámetro principal: mínimo de reglas que deben cumplirse 
# Rango válido: 1–8. Recomendado: 5 (conservador-moderado), 4 (agresivo), 6 (muy conservador)
MIN_RULES_TO_SIGNAL = 5
# Umbrales RSI 
RSI_OVERSOLD    = 35 # 35
RSI_OVERBOUGHT  = 70 # 65
# Umbrales Estocástico
STOCH_OVERSOLD  = 25
STOCH_OVERBOUGHT = 75
# Umbrales de posición dentro del canal BB
BB_ENTRY_SELL_MIN  = 0.75   # Para VENDER: precio debe estar en el > 75% del canal
BB_ENTRY_BUY_MAX   = 0.25   # Para COMPRAR: precio debe estar en el < 25% del canal
BB_WAIT_SELL_MIN   = 0.45   # Por debajo de esto en venta  → esperar subida
BB_WAIT_BUY_MAX    = 0.55   # Por encima de esto en compra → esperar bajada
# VPVMA
BANDWITH=0.1 # 0.1 filtro para reducir señales falsas (default 0.1)
# Volatility
VOLATILITY_PERIOD = 20
# Hull MA
HMA_FAST = 21
HMA_SLOW = 55
# Breakout structure
BREAKOUT_N = 20

import pandas as pd
def calc_ema(series: pd.Series, period: int) -> pd.Series:
    return series.ewm(span=period, adjust=False).mean()


def calc_rsi(close: pd.Series, period: int = 14) -> pd.Series:
    delta = close.diff()
    gain  = delta.clip(lower=0)
    loss  = -delta.clip(upper=0)
    avg_gain = gain.ewm(com=period - 1, min_periods=period).mean()
    avg_loss = loss.ewm(com=period - 1, min_periods=period).mean()
    rs  = avg_gain / avg_loss.replace(0, np.nan)
    rsi = 100 - (100 / (1 + rs))
    return rsi


def calc_macd(close: pd.Series, fast: int, slow: int, signal: int):
    ema_fast   = calc_ema(close, fast)
    ema_slow   = calc_ema(close, slow)
    macd_line  = ema_fast - ema_slow
    signal_line = calc_ema(macd_line, signal)
    histogram   = macd_line - signal_line
    return macd_line, signal_line, histogram


def calc_bollinger(close: pd.Series, period: int, std_mult: float):
    mid   = close.rolling(period).mean()
    std   = close.rolling(period).std()
    upper = mid + std_mult * std
    lower = mid - std_mult * std
    return upper, mid, lower


def calc_stochastic(high: pd.Series, low: pd.Series, close: pd.Series,
                    k_period: int, slowing: int, d_period: int):
    lowest_low   = low.rolling(k_period).min()
    highest_high = high.rolling(k_period).max()
    denom = (highest_high - lowest_low).replace(0, np.nan)
    
    raw_k = 100 * (close - lowest_low) / denom  # %K crudo
    k     = raw_k.rolling(slowing).mean()        # %K suavizado (Slowing)
    d     = k.rolling(d_period).mean()           # %D
    return k, d


def calc_atr(high: pd.Series, low: pd.Series, close: pd.Series,
             period: int) -> pd.Series:
    prev_close = close.shift(1)
    tr = pd.concat([
        high - low,
        (high - prev_close).abs(),
        (low  - prev_close).abs()
    ], axis=1).max(axis=1)
    return tr.ewm(com=period - 1, min_periods=period).mean()


def calc_obv(close: pd.Series, volume: pd.Series) -> pd.Series:
    direction = np.sign(close.diff()).fillna(0)
    return (direction * volume).cumsum()


def calc_adx(high, low, close, period=14):
    tr = calc_atr(high, low, close, period) 

    up_move   = high.diff()
    down_move = -low.diff()

    plus_dm  = np.where((up_move > down_move) & (up_move > 0), up_move, 0)
    minus_dm = np.where((down_move > up_move) & (down_move > 0), down_move, 0)

    plus_dm  = pd.Series(plus_dm,  index=close.index)
    minus_dm = pd.Series(minus_dm, index=close.index)

    atr      = tr
    plus_di  = 100 * plus_dm.ewm(com=period-1).mean()  / atr
    minus_di = 100 * minus_dm.ewm(com=period-1).mean() / atr

    dx  = 100 * (plus_di - minus_di).abs() / (plus_di + minus_di)
    adx = dx.ewm(com=period-1).mean()

    return adx, plus_di, minus_di


def calc_bb_position(df):
    """
    Retorna la posición relativa del precio dentro del canal BB.
      0.0 → precio en la banda inferior
      0.5 → precio en la media (mitad del canal)
      1.0 → precio en la banda superior
    """
    return (df["Close"] - df["BB_lower"]) / (df["BB_upper"] - df["BB_lower"] + 1e-9)


def calculate_vpvma(df, window_fast=12, window_slow=26, window_sign=9):
    """
    VPVMA - Volume Price Volatility Moving Average
    Derivado del MACD por Pat Tong Chio (2022)
    
    Parámetros:
        df           : DataFrame con columnas Open, High, Low, Close, Volume
        window_fast  : período corto (default 12)
        window_slow  : período largo (default 26)
        window_sign  : período de señal (default 9)
        bandwidth    : filtro para reducir señales falsas (default 0.1)
    """
    df["TP"] = (df["High"] + df["Low"] + df["Close"]) / 3

    df["TP_x_Vol"] = df["TP"] * df["Volume"]

    df["SVWMA"] = (
        df["TP_x_Vol"].rolling(window=window_fast).sum() /
        df["Volume"].rolling(window=window_fast).sum()
    )

    df["LVWMA"] = (
        df["TP_x_Vol"].rolling(window=window_slow).sum() /
        df["Volume"].rolling(window=window_slow).sum()
    )

    df["DV"] = df[["Open", "High", "Low", "Close"]].std(axis=1)

    df["SVWMA_x_DV"] = df["SVWMA"] * df["DV"]
    df["LVWMA_x_DV"] = df["LVWMA"] * df["DV"]

    df["ESVMap"] = df["SVWMA_x_DV"].ewm(span=window_fast, adjust=False).mean()
    df["ELVMap"] = df["LVWMA_x_DV"].ewm(span=window_slow, adjust=False).mean()

    df["VPVMA"]  = df["ESVMap"] - df["ELVMap"]
    df["VPVMAS"] = df["VPVMA"].rolling(window=window_sign).mean()
    df.drop(columns = ["SVWMA", "TP", "TP_x_Vol", "SVWMA", "DV", "SVWMA_x_DV", "LVWMA_x_DV", "ESVMap", "ELVMap"], inplace = True)    


    return df


def evaluate_entry_quality(signal: str, bb_pos: float) -> dict:
    """
    Evalúa si el precio está en una zona ÓPTIMA para entrar o si es necesario esperar el REBOTE. 
    Se valida que no esteen la mitad del canal, sino en zonas XTREMs (superior para venta, inferior para compra).
    
    Zonas para VENDER (bb_pos):
    🟢 ZONA ÓPTIMA VENTA    (>= 0.75)      
    🟡 ZONA ACEPTABLE VENTA (0.55 – 0.75)  
    🔴 ESPERAR SUBIDA       (< 0.55)       

    Zonas para COMPRAR (bb_pos):
    🔴 ESPERAR BAJADA       (> 0.55)      
    🟡 ZONA ACEPTABLE COMPRA (0.25 – 0.45) 
    🟢 ZONA ÓPTIMA COMPRA   (<= 0.25) 
    """
    pct = bb_pos * 100 

    if "🔴" in signal:
        if bb_pos >= BB_ENTRY_SELL_MIN:
            return {
                "entry_ok":   True,
                "advice": "🔴",
                "entry_zone":     f"🔴",
                "bb_pct":     pct,
            }
        elif bb_pos >= BB_WAIT_SELL_MIN:
            return {
                "entry_ok":   False,
                "advice": "🟡", # Nivel aceptable, mejor esperar 
                "entry_zone":     (f"🟡 > {BB_ENTRY_SELL_MIN*100:.0f}%"),
                "bb_pct":     pct,
            }
        else:
            return {
                "entry_ok":   False,
                "advice": "⚪",
                "entry_zone":     (f"⚪ > {BB_ENTRY_SELL_MIN*100:.0f}%"), 
                "bb_pct":     pct,
            }

    elif "🟢" in signal:
        if bb_pos <= BB_ENTRY_BUY_MAX:
            return {
                "entry_ok":   True,
                "advice": "🟢",
                "entry_zone":     f"🟢",
                "bb_pct":     pct,
            }
        elif bb_pos <= BB_WAIT_BUY_MAX:
            return {
                "entry_ok":   False,
                "advice": "🟡",  
                "entry_zone":     (f"🟡 < {BB_ENTRY_BUY_MAX*100:.0f}%"),
                "bb_pct":     pct,
            }
        else:
            return {
                "entry_ok":   False,
                "advice": "⚪",
                "entry_zone":     (f"⚪ < {BB_ENTRY_BUY_MAX*100:.0f}%"),
                "bb_pct":     pct,
            }

    # Para ESPERAR no aplica
    return {"entry_ok": False, "entry_zone": "", "advice": "", "bb_pct": pct}

def calc_consistencia(close: pd.Series) -> pd.Series:
    prev26 = close.shift(26)
    prev52 = close.shift(52)

    return prev26, prev52

def build_indicators(df: pd.DataFrame) -> pd.DataFrame:
    c, h, l, v = df["Close"], df["High"], df["Low"], df["Volume"]

    df["EMA20"]  = calc_ema(c, EMA_SHORT)
    df["EMA50"]  = calc_ema(c, EMA_MID)
    df["EMA200"] = calc_ema(c, EMA_LONG)

    df["RSI"] = calc_rsi(c, RSI_PERIOD)

    df["MACD"], df["MACD_signal"], df["MACD_hist"] = calc_macd(
        c, MACD_FAST, MACD_SLOW, MACD_SIGNAL
    )

    df["BB_upper"], df["BB_mid"], df["BB_lower"] = calc_bollinger(c, BB_PERIOD, BB_STD)

    df["STOCH_K"], df["STOCH_D"] = calc_stochastic(h, l, c, STOCH_K, STOCH_SLOWING, STOCH_D)

    df["ATR"] = calc_atr(h, l, c, ATR_PERIOD)

    df["OBV"]     = calc_obv(c, v)
    df["OBV_EMA"] = calc_ema(df["OBV"], 20)

    df["ADX"], df["ADX+DI"], df["ADX-DI"] = calc_adx(h, l, c, period=14)

    df["prev26"], df["prev52"] = calc_consistencia(c)

    df = calculate_vpvma(df)

    df["VWAP"] = (df["Close"] * df["Volume"]).cumsum() / df["Volume"].cumsum()

    df["spread"] = (df["High"] - df["Low"]) / df["Close"]
    
    df["vol_ma"] = df["Volume"].rolling(20).mean()
    df["volume_increasing"] = df["Volume"] > df["vol_ma"]
    
    df["order_flow"] = np.where(df["Close"] > df["Open"], "buy", "sell")

    df["volatility"] = df["Close"].pct_change().rolling(VOLATILITY_PERIOD).std()

    # df.dropna(inplace=True)
    return df


def fmt(n, decimals=4) -> str:
    if n is None:
        return "N/A"
    fmt_str = f"{{:,.{decimals}f}}"
    return fmt_str.format(n)


def compute_tp_sl(price: float, atr: float, direction: str) -> tuple:
    """
    Retorna (take_profit, stop_loss) basados en ATR.
    direction: 'BUY' o 'SELL'
    """
    if direction == "🟢":
        sl = price - ATR_MULT_SL * atr
        tp = price + ATR_MULT_TP * atr
    else:  # SELL
        sl = price + ATR_MULT_SL * atr
        tp = price - ATR_MULT_TP * atr
    return tp, sl

def analyze_symbol(symbol: str, INTERVAL:int, LOOKBACK_DAYS:int,  DAY = None, HOUR = None) -> dict:
    # symbol = "CVS" # usar antes del 15/04 ya que 3 estrategias se alinean para sell
    # LOOKBACK_DAYS = 40
    # INTERVAL = "1h"

    for _ in range(3):
        try:
            df = yf.download(
                symbol,
                period=f"{LOOKBACK_DAYS}d",
                interval=INTERVAL,
                progress=False,
                auto_adjust=True,
            )
        except Exception as e:
            print(f"ERROR: {symbol} - {e}")
            return pd.DataFrame()

        if len(df) > 0:
            break   

    if df.empty or len(df) < EMA_LONG + 10:
        print(f"ERROR: {symbol} - len(df) < {EMA_LONG + 10}")
        return pd.DataFrame()

    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    if len(df) < 2:
        print(f"ERROR: {symbol} - len(df) < 2")
        return pd.DataFrame()
    
    df["date"] = df.index
    df = build_indicators(df.copy())
    bb_pos   = calc_bb_position(df)

    if DAY != None:
        df["hour"] = df["date"].dt.strftime("%H").astype(int)
        df["day"] = df["date"].dt.strftime("%Y-%m-%d")

        df = df[(df["day"] <= DAY) & (df["hour"] <= HOUR)]


    # ------------------------------- SEÑALES -------------------------------#
    
    #---------------------------------- VERSIONES MACD + RSI
    # ----------- 1.1 MACD + RSI exigente
    """
    RSI bajo durante varias velas = presión vendedora estructural
    MACD cruzando = inicio del cambio
    Es un detector de “agotamiento + giro temprano”
    """
    df["rsi_below_35"] = (df["RSI"] < 30).astype(int)
    df["rsi_above_65"] = (df["RSI"] > 70).astype(int)

    df["count_men"] = df["rsi_below_35"].rolling(6, min_periods = 1).sum()
    df["count_may"] = df["rsi_above_65"].rolling(6, min_periods = 1).sum()

    df["macd_signal"] = np.where(
        df["MACD"] > df["MACD_signal"], 1,
        np.where(df["MACD"] < df["MACD_signal"], -1, 0)
    )

    # El umbral 4 originalmente era 6 (AJUSTE para que salgan señales)
    df["sign_macd_rsi_robust"] = np.where(
        (df["count_men"] >= 4) & (df["macd_signal"] == 1), "🟢",
        np.where(
            (df["count_may"] >= 4) & (df["macd_signal"] == -1), "🔴",
            ""
        )
    )
    del df["rsi_below_35"], df["rsi_above_65"], df["count_men"], df["count_may"], df["macd_signal"]


    #-----------1.2 MACD + RSI no exigente
    df["str_macd_rsi_simple"] = np.where((df["MACD"] > df["MACD_signal"]) & (df["RSI"] < 30), "🟢",
                                    np.where((df["MACD"] < df["MACD_signal"]) & (df["RSI"] > 70), "🔴", ""))

    #-----------1.1.2 MACD + RSI no exigente + confirmacion breakout
    df["str_macd_rsi_simple_break"] = np.where((df["str_macd_rsi_simple"] == "🟢") & (df["Close"] > df["High"].shift(1)), "🟢",
                                          np.where((df["str_macd_rsi_simple"] == "🔴") & (df["Close"] < df["Low"].shift(1)), "🔴", ""))

    # ---------- 2.2 MACD + RSI exigente + filtro de posición en BB
    df["bb_pos"] = bb_pos
    df["str_macd_rsi_simple_bb_pos"] = np.where((df["str_macd_rsi_simple"] == "🟢") & (df["bb_pos"] < 0.2), "🟢",
                                np.where((df["str_macd_rsi_simple"] == "🔴") & (df["bb_pos"] > 0.8), "🔴", ""))

    #---------- 3.1 RSI + Bollinger
    df["str_macd_rsi_bb"] = np.where((df["RSI"] < 30) & (df["Close"] < df["BB_lower"]), "🟢", 
                                   np.where((df["RSI"] > 70) & (df["Close"] > df["BB_upper"]), "🔴", ""))
    
    #---------- 3.2 RSI + Bollinger + Tendencia EMA
    df["str_macd_rsi_bb_ema"] = np.where((df["RSI"] < 30) & (df["Close"] < df["BB_lower"]) & (df["Close"] > df["EMA200"]), "🟢", 
                                   np.where((df["RSI"] > 70) & (df["Close"] > df["BB_upper"]) & (df["Close"] < df["EMA200"]), "🔴", ""))
    


    #---------------------------------- VERSIONES ADX
    #---------- 1 ADX + DI
    df["str_adx"] = np.where((df["ADX"] > 25) & (df["ADX+DI"] > df["ADX-DI"]), "🟢", 
                     np.where((df["ADX"] > 25) & (df["ADX-DI"] > df["ADX+DI"]), "🔴", ""))
    
    #---------- 1.1 ADX + DI + Pendiente tendencia ADX
    # ADX > 25 puede venir de un movimiento viejo o ya agotado.
    # ADX creciente (ADX[t] > ADX[t-1])
    df["str_adx_crec"] = np.where((df["str_adx"] == "🟢") & (df["ADX"] > df["ADX"].shift(1)), "🟢",
                          np.where((df["str_adx"] == "🔴") & (df["ADX"] > df["ADX"].shift(1)), "🔴", ""))


    #---------- 1.1.2 ADX + DI + Pendiente tendencia ADX
    df["str_adx_crec_rsi"] = np.where((df["str_adx"] == "🟢") & (df["ADX"] > df["ADX"].shift(1)) & (df["RSI"] < 45), "🟢",
                              np.where((df["str_adx"] == "🔴") & (df["ADX"] > df["ADX"].shift(1)) & (df["RSI"] > 55), "🔴", ""))

    #---------- 1.1.3 ADX + DI + Pendiente tendencia ADX
    df["str_adx_rsi"] = np.where((df["str_adx"] == "🟢") & (df["RSI"] < 40), "🟢",
                         np.where((df["str_adx"] == "🔴") & (df["RSI"] > 60), "🔴", ""))

    df["str_adx_rsi_robust"] = np.where((df["str_adx"] == "🟢") & (df["RSI"] < 30), "🟢",
                                np.where((df["str_adx"] == "🔴") & (df["RSI"] > 70), "🔴", ""))
    
    df["str_adx_rsi_xtrem"] = np.where((df["str_adx"] == "🟢") & (df["RSI"] < 20), "🟢",
                               np.where((df["str_adx"] == "🔴") & (df["RSI"] > 80), "🔴", ""))



    # ---------- 2. VPVMA (volumen ponderado dinámico)
    """
    Qué mide: Precio ponderado por volumen vs su media
    Es parecido a VWAP, pero más “reactivo”
    Cruce = cambio en el equilibrio entre compradores/vendedores reales
    Detecta: “quién está ganando con dinero real”
    Riesgo: Sensible a spikes de volumen
    """

    signbuy_base = (
        (df["VPVMA"] > (1 + BANDWITH) * df["VPVMAS"]) &
        (df["VPVMA"].shift(1) <= df["VPVMAS"].shift(1))
    )

    signsell_base = (
        (df["VPVMA"] < (1 - BANDWITH * 2) * df["VPVMAS"]) &
        (df["VPVMA"].shift(1) >= df["VPVMAS"].shift(1))
    )

    window = 2
    recent_buy = signbuy_base.rolling(window).max().fillna(0).astype(bool)
    recent_sell = signsell_base.rolling(window).max().fillna(0).astype(bool)

    df["str_vpvma"] = np.where(recent_buy, "🟢", np.where(recent_sell, "🔴", ""))
    # vpvma = df["vpvma_sign"].iloc[-1]


    # ----------- 3. Sinergia
    """
    “solo opero cuando TODO está alineado”
    Riesgo
    Puede filtrar demasiado (pocas señales)
    """
    long_cond = (
        (df["Close"] > df["EMA20"]) &
        (df["RSI"] > 55) &
        (df["MACD"] > df["MACD_signal"]) &
        (df["Close"] > df["VWAP"]) &
        (df["spread"] < 0.01) &
        (df["volume_increasing"]) &
        (df["order_flow"] == "buy")
    )

    short_cond = (
        (df["Close"] < df["EMA20"]) &
        (df["RSI"] < 45) &
        (df["MACD"] < df["MACD_signal"]) &
        (df["Close"] < df["VWAP"]) &
        (df["spread"] < 0.01) &
        (df["volume_increasing"]) &
        (df["order_flow"] == "sell")
    )

    df["str_sinergia"] = ""
    df.loc[long_cond, "str_sinergia"] = "🟢"
    df.loc[short_cond, "str_sinergia"] = "🔴"
    # sinergia = df["str_sinergia"].iloc[-1]
    del df["order_flow"], df["volume_increasing"], df["spread"]


    # ----------- 4. RSI mean reversion + Bollinger
    """
    Qué mide
    Desviación XTREM del precio respecto a la media
    Se predice que el precio hace una reversion a la media
    Interpretación: “el mercado se estiró demasiado, va a regresar”
    Riesgo: En tendencias fuertes, el precio sigue “rompiendo la banda”
    """

    df["str_rsi_bb_rever"] = np.where((df['RSI'] < 30) & (df['Close'] <= df['Low']), "🟢", 
                             np.where((df['RSI'] > 70) & (df['Close'] >= df['High']), "🔴", ""))
    # reversionmean = df["str_rsi_bb_rever"].iloc[-1]
    

    # ----------- 5. Trend following: MA + MACD. 
    """
    Qué mide: Dirección + confirmación de momentum
    Interpretación: “me subo a la ola, no intento predecirla”
    Riesgo: Entradas tardías
    """
    
    df['trend'] = np.where(df['Close'] > df['EMA200'], 1, -1)

    df['long_signal'] = (
        (df['EMA50'] > df['EMA200']) &
        (df['MACD'] > df['MACD_signal']) &
        (df['trend'] == 1)
    )

    df['short_signal'] = (
        (df['EMA50'] < df['EMA200']) &
        (df['MACD'] < df['MACD_signal']) &
        (df['trend'] == -1)
    )

    # Evitar lookahead bias
    df['long_signal'] = df['long_signal'].shift(1)
    df['short_signal'] = df['short_signal'].shift(1)

    df["str_ma_macd"] = ""
    df.loc[df['long_signal'] == True, "str_ma_macd"] = "🟢"
    df.loc[df['short_signal'] == True, "str_ma_macd"] = "🔴"
    del df["long_signal"], df["short_signal"]

    # trendfollow = df["str_ma_macd"].iloc[-1]


    # ----------- 6. Squeeze + Keltner + Bollinger (str_squ_kel_bb)
    """
    Qué mide: Contracción de volatilidad
    Interpretación: “el mercado está comprimido → se viene explosión”
    Riesgo: No siempre predice dirección
    """

    length = 20
    mult_bb = 2.0
    mult_kc = 1.5

    # Bollinger Bands
    df['ma_20'] = df['Close'].rolling(window=length).mean()
    df['std_20'] = df['Close'].rolling(window=length).std()
    df['bb_upper'] = df['ma_20'] + (mult_bb * df['std_20'])
    df['bb_lower'] = df['ma_20'] - (mult_bb * df['std_20'])

    # Keltner Channels (Usando ATR para la banda)
    df['keltner_channels'] = np.maximum(df['High'] - df['Low'], 
                            np.maximum(abs(df['High'] - df['Close'].shift(1)), 
                                        abs(df['Low'] - df['Close'].shift(1))))
    df['atr_20'] = df['keltner_channels'].rolling(window=length).mean()
    df['kc_upper'] = df['ma_20'] + (mult_kc * df['atr_20'])
    df['kc_lower'] = df['ma_20'] - (mult_kc * df['atr_20'])

    # Lógica del Squeeze
    df['sign_squeeze'] = (df['bb_upper'] < df['kc_upper']) & (df['bb_lower'] > df['kc_lower'])

    # Momentum (Filtro simple para dirección del squeeze)
    df['momentum'] = df['Close'] - df['Close'].shift(length)

    df["str_squ_kel_bb"] = np.where((df['sign_squeeze'] == True) & (df['momentum'] > 0), "🟢",  
                           np.where((df['sign_squeeze'] == True) & (df['momentum'] < 0), "🔴", ""))
    
    # SE CAMBIO MOMENTUM POR ADX
    df["str_squ_kel_bb"] = np.where((df['sign_squeeze'] == True) & (df['str_adx'] == "🟢"), "🟢",  
                           np.where((df['sign_squeeze'] == True) & (df['str_adx'] == "🔴"), "🔴", ""))
    
    del df["momentum"], df["bb_upper"], df["bb_lower"], df["kc_upper"], df["kc_lower"], df["keltner_channels"]


    # ---------- 7. Hull MA
    """
    Qué mide: Tendencia suavizada pero rápida
    Interpretación: Más rápida que EMA, menos ruido
    Detecta: cambios de tendencia más temprano
    """
    def calculate_hull_ma(series, n):
        def wma(s, period):
            weights = np.arange(1, period + 1)
            return s.rolling(period).apply(lambda x: np.dot(x, weights) / weights.sum(), raw=True)
        
        half_n = int(n / 2)
        sqrt_n = int(np.sqrt(n))
        
        val = 2 * wma(series, half_n) - wma(series, n)
        hma = wma(val, sqrt_n)
        return hma

    df['hma_fast'] = calculate_hull_ma(df['Close'], HMA_FAST)
    df['hma_slow'] = calculate_hull_ma(df['Close'], HMA_SLOW)

    df['sign_hullma'] = np.where(df['hma_fast'] > df['hma_slow'], "🟢", 
                        np.where(df['hma_fast'] < df['hma_slow'], "🔴", ""))
    # hma = df['hma_sign'].iloc[-1]
    del df["hma_fast"], df["hma_slow"]



    # ----------- 1. Golden Cross y Death Cross 
    """
    Qué mide: Cambio estructural de tendencia
    Interpretación: Es una señal “lenta pero fuerte”
    Riesgo: Muy retrasada
    """
    golden_base = (
        (df["EMA20"] > df["EMA50"]) &
        (df["EMA20"].shift(1) <= df["EMA50"].shift(1))
    )

    death_base = (
        (df["EMA20"] < df["EMA50"]) &
        (df["EMA20"].shift(1) >= df["EMA50"].shift(1))
    )

    window = 2
    recent_golden = golden_base.rolling(window).max().fillna(0).astype(bool)
    recent_death = death_base.rolling(window).max().fillna(0).astype(bool)

    df["sign_gold"] = np.where(recent_golden, "🟢", np.where(recent_death, "🔴", ""))


    # ----------- 2. Volumen — OBV 
    """
    Qué mide: Flujo acumulado de volumen
    Interpretación: “el dinero entra antes que el precio”
    # Volumen confirma presión compradora o vendedora
    """
    df["sign_obv"] =  np.where(df["OBV"] > df["OBV_EMA"],  "🟢", 
                      np.where(df["OBV"] < df["OBV_EMA"],  "🔴", ""))
    # volume = df["volume_sign"].iloc[-1]
        

    # ----------- 3. 3-EMA alignment
    """
    Qué mide: Jerarquía de tendencias
    Interpretación: mercado ordenado vs caótico
    """
    df['str_3ema'] = np.where((df['EMA20'] > df['EMA50']) & (df['EMA50'] > df['EMA200']), "🟢",  np.where((df['EMA20'] < df['EMA50']) & (df['EMA50'] < df["EMA200"]), "🔴", ""))
    # ema3 = df['str_3ema'].iloc[-1]


    # ----------- 4. Sobrecompra y sobreventa RSI
    df["sign_rsi30"] = np.where((df["RSI"] < 30), "🟢", np.where((df["RSI"] > 70), "🔴", ""))
    df["sign_rsi20"] = np.where((df["RSI"] < 20), "🟢", np.where((df["RSI"] > 80), "🔴", ""))
    df["sign_rsi15"] = np.where((df["RSI"] < 15), "🟢", np.where((df["RSI"] > 85), "🔴", ""))


    # ----------- 5. ADX + DI - Direccion y fuerza tendencia
    """
    Qué mide: Fuerza + dirección
    Interpretación: diferencia entre tendencia real vs ruido
    """
    def classify(row):
        adx_val   = row["ADX"]
        p_di      = row["ADX+DI"]
        m_di      = row["ADX-DI"]
        precio    = row["Close"]
        ema200    = row["EMA200"]

        if pd.isna(adx_val) or pd.isna(p_di) or pd.isna(m_di):
            return "- 0"

        #  Mercado lateral: ADX muy bajo, sin dirección clara
        if adx_val < 15:
            return ""

        # Determinar dirección
        alcista = (p_di > m_di) and (precio > ema200)
        bajista = (m_di > p_di) and (precio < ema200)

        # Caso borde: DI's apuntan contra EMA200 (conflicto de señales)
        if not alcista and not bajista:
            if p_di > m_di:
                return "▲ 1"   # alcista pero bajo EMA200
            elif m_di > p_di:
                return "▼ 1"   # bajista pero sobre EMA200
            else:
                return ""

        # Clasificar por fuerza del ADX
        if alcista:
            if adx_val >= 70:
                return "▲ 5 SUPER XTREM"
            elif adx_val >= 60:
                return "▲ 4 XTREM"
            elif adx_val >= 40:
                return "▲ 3"
            elif adx_val >= 25:
                return "▲ 2"
            else:             
                return "▲ 1"
        if bajista:
            if adx_val >= 70:
                return "▼ 5 SUPER XTREM"
            elif adx_val >= 60:
                return "▼ 4 XTREM"
            elif adx_val >= 40:
                return "▼ 3"
            elif adx_val >= 25:
                return "▼ 2"
            else:
                return "▼ 1"

        return ""

    df["trend"] = df.apply(classify, axis=1)


    # ----------- 6. Breakout de estructura
    """
    BUY: rompe el máximo de N velas
    SELL: rompe el mínimo de N velas    
    Qué mide: no mide “indicadores”… mide estructura pura de precio.
    Máximos y mínimos recientes = zonas donde el mercado ya tomó decisiones
    Cuando se rompen → cambia el equilibrio
    Interpretación profunda: Un breakout es literalmente:
    “los compradores (o vendedores) absorbieron toda la liquidez disponible en ese nivel”
    No es solo un cruce… es consumo de órdenes pendientes

    Breakout sin volumen → sospechoso
    Breakout + volumen → institucional
    """

    max_bars_since_break = 4  # límite de velas válidas
    strength_factor = 0.4

    df["hh"] = df["High"].rolling(BREAKOUT_N).max()
    df["ll"] = df["Low"].rolling(BREAKOUT_N).min()

    df["range"] = df["High"] - df["Low"]

    df["break_up"] = (
        (df["Close"] > df["hh"].shift(1)) &
        (df["Close"].shift(1) <= df["hh"].shift(2))
    )

    df["break_down"] = (
        (df["Close"] < df["ll"].shift(1)) &
        (df["Close"].shift(1) >= df["ll"].shift(2))
    )

    df["strong_break_up"] = (
        df["break_up"] &
        (df["Close"] > df["hh"].shift(1) + strength_factor * df["range"])
    )

    df["strong_break_down"] = (
        df["break_down"] &
        (df["Close"] < df["ll"].shift(1) - strength_factor * df["range"])
    )

    df["bars_since_break_up"] = np.where(df["strong_break_up"], 0, np.nan)
    df["bars_since_break_down"] = np.where(df["strong_break_down"], 0, np.nan)

    df["bars_since_break_up"] = df["bars_since_break_up"].ffill()
    df["bars_since_break_down"] = df["bars_since_break_down"].ffill()

    df["bars_since_break_up"] = df["bars_since_break_up"].groupby(df["bars_since_break_up"].notna().cumsum()).cumcount()

    df["bars_since_break_down"] = df["bars_since_break_down"].groupby(df["bars_since_break_down"].notna().cumsum()).cumcount()

    df["str_breakout"] = np.where((df["bars_since_break_up"] <= max_bars_since_break), "🟢",
                         np.where((df["bars_since_break_down"] <= max_bars_since_break), "🔴", ""))
    
    del df["hh"], df["ll"], df["range"], df["break_up"], df["break_down"], df["strong_break_up"], df["strong_break_down"], df["bars_since_break_up"], df["bars_since_break_down"]

    # breakout = df["msb_sign"].iloc[-1]
    

    # ----------- 8. Pullback + EMA
    """
    Qué mide: Retroceso dentro de tendencia
    Interpretación: no persigo el precio, espero descuento
    Es lo que hacen traders institucionales: Compran en retrocesos. No en rompimientos emocionales

    Mejora el ratio riesgo/beneficio brutalmente. Reduce drawdowns
    """
    df["str_pullb_ema"] = np.where(
        (df["Close"] > df["EMA50"]) &
        (df["Low"] <= df["EMA20"]) &
        (df["Close"] > df["EMA20"]),
        "🟢",
        np.where(
            (df["Close"] < df["EMA50"]) &
            (df["High"] >= df["EMA20"]) &
            (df["Close"] < df["EMA20"]),
            "🔴", ""))
    
    # pullback = df["pullback_sign"].iloc[-1]


    # ----------- 9. Divergencia RSI - agotamiento 
    """
    Qué mide: Desacople entre precio y momentum
    Interpretación: el precio sigue subiendo… pero con menos fuerza interna. Es como un coche subiendo una montaña 
    con el motor fallando.

    Muy potente en techos/suelos
    Mejor en temporalidades altas

    Precio hace lower low, RSI no → BUY
    Precio hace higher high, RSI no → SELL
    """
    df["price_diff"] = df["Close"].diff()
    df["rsi_diff"] = df["RSI"].diff()

    df["str_rsi_div"] =np.where((df["price_diff"] < 0) & (df["rsi_diff"] > 0), "🟢",
                       np.where((df["price_diff"] > 0) & (df["rsi_diff"] < 0), "🔴", ""))
    del df["price_diff"], df["rsi_diff"]


    # ----------- 10. VWAP Deviation
    """
    Muy lejos del VWAP → reversión probable
    Qué mide: Distancia respecto al precio promedio ponderado
    Interpretación: qué tan “caro” o “barato” está el precio en el día

    Institucionales usan VWAP como referencia. Grandes desviaciones → reversión probable
    """
    df["vwap_dev"] = (df["Close"] - df["VWAP"]) / df["VWAP"]

    df["sign_vwap_devi"] = np.where(df["vwap_dev"] < -0.02, "🟢", np.where(df["vwap_dev"] > 0.02, "🔴", ""))
    # vwap_dev = df["vwap_dev_sign"].iloc[-1]
    del df["vwap_dev"]


    # 11. ----------- Support / Resistance dinámico 
    """
    Qué mide: Zonas donde el precio ha reaccionado antes
    Interpretación: memoria del mercado. Los traders recuerdan esos niveles… y actúan ahí

    Más fuerte si coincide con volumen
    Más fuerte si coincide con estructura
    """
    n = 50
    lookback_break = 5  # máximo de velas desde el rompimiento

    df["resistance"] = df["High"].rolling(n).max()
    df["support"] = df["Low"].rolling(n).min()

    df["break_up"] = (df["Close"] > df["resistance"].shift(1))
    df["break_down"] = (df["Close"] < df["support"].shift(1))

    df["bars_since_break_up"] = (~df["break_up"]).cumsum() - (~df["break_up"]).cumsum().where(df["break_up"]).ffill()
    df["bars_since_break_down"] = (~df["break_down"]).cumsum() - (~df["break_down"]).cumsum().where(df["break_down"]).ffill()

    df["str_supo_resi"] = np.where(df["bars_since_break_up"] <= lookback_break, "🔴", 
                             np.where(df["bars_since_break_down"] <= lookback_break, "🟢", ""))
    # sr = df["supor_resis_sign"].iloc[-1]
    del df["resistance"], df["support"], df["break_up"], df["break_down"], df["bars_since_break_up"], df["bars_since_break_down"]


    # ----------- 13. Heikin Ashi Trend
    """
    Suaviza velas → mejor lectura de tendencia
    Qué mide: Tendencia suavizada
    Interpretación: elimina ruido emocional del mercado

    Ideal para mantener posiciones
    No para entradas precisas
    """
    df["ha_close"] = (df["Open"] + df["High"] + df["Low"] + df["Close"]) / 4
    df["ha_open"]  = (df["Open"].shift(1) + df["Close"].shift(1)) / 2

    df["sign_heikin"] = np.where(df["ha_close"] > df["ha_open"], "🟢",
                        np.where(df["ha_close"] < df["ha_open"], "🔴", ""))
    # ha_signal = df["ha_sign"].iloc[-1]
    del df["ha_close"], df["ha_open"]


    # ----------- 14. Momentum puro (Rate of Change)
    """
    Mide velocidad del precio (aceleración)
    Qué mide: velocidad del precio
    Interpretación: qué tan rápido se mueve el mercado

    Momentum precede a tendencia
    """
    n = 10
    df["ROC"] = df["Close"].pct_change(n)

    df["sign_momentum"] = np.where(df["ROC"] > 0.02, "🟢",
                          np.where(df["ROC"] < -0.02, "🔴", ""))
    # momentum = df["momentum_sign"].iloc[-1]


    # # ----------- 15. Volumen anómalo (Volume Spike)
    """
    Detecta actividad institucional o eventos
    Qué mide: volumen anormal
    Interpretación: actividad institucional o evento importante

    Confirmación clave de movimientos
    """
    df["vol_mean"] = df["Volume"].rolling(20).mean()

    df["sign_volume_anormal"] = np.where(df["Volume"] > 2 * df["vol_mean"], "✅", "")
    # volume_spike = df["volspike_sign"].iloc[-1]
    del df["vol_mean"]


    # ----------- 16. Range Compression (antes de breakout)
    """
    Detecta consolidación XTREM
    Qué mide: reducción XTREM del rango
    Interpretación: mercado “apretado” antes de moverse

    Precede breakouts
    """
    df["range"] = df["High"] - df["Low"]
    df["range_mean"] = df["range"].rolling(20).mean()

    df["sign_compress_xtrem"] = np.where(df["range"] < 0.5 * df["range_mean"], "✅", "")
    # compression = df["compress_sign"].iloc[-1]
    del df["range"], df["range_mean"]


    # ----------- 17. EMA Distance (sobre-extensión)
    """
    Detecta cuando el precio está demasiado lejos de la media
    Qué mide: sobre-extensión del precio
    Interpretación: el precio se alejó demasiado de su equilibrio

    Evita entrar tarde
    Ideal para contrarian trades
    """
    df["ema_dist"] = (df["Close"] - df["EMA50"]) / df["EMA50"]

    df["sign_sobreext"] = np.where(df["ema_dist"] > 0.03, "🔴",
                          np.where(df["ema_dist"] < -0.03, "🟢", ""))
    # ema_distance = df["emadist_sign"].iloc[-1]


    # ----------- 18. Supertrend (ATR-based dynamic S/R)
    """
    Genera soporte/resistencia dinámico basado en ATR. Excelente complemento al HMA porque confirma dónde está el soporte, no solo la dirección.
    """
    def calc_supertrend(df, period=10, multiplier=3.0):
        hl2 = (df["High"] + df["Low"]) / 2
        atr = df["ATR"]

        upper = hl2 + (multiplier * atr)
        lower = hl2 - (multiplier * atr)

        supertrend = pd.Series(index=df.index, dtype=float)
        direction = pd.Series(index=df.index, dtype=int)

        supertrend.iloc[0] = upper.iloc[0]
        direction.iloc[0] = 1

        for i in range(1, len(df)):
            prev_st = supertrend.iloc[i-1]
            prev_dir = direction.iloc[i-1]

            up = upper.iloc[i]
            dn = lower.iloc[i]

            if prev_dir == 1:
                up = min(up, prev_st)
            else:
                dn = max(dn, prev_st)

            if df["Close"].iloc[i] > up:
                direction.iloc[i] = -1
            elif df["Close"].iloc[i] < dn:
                direction.iloc[i] = 1
            else:
                direction.iloc[i] = prev_dir

            supertrend.iloc[i] = dn if direction.iloc[i] == -1 else up

        df["supertrend"] = supertrend
        df["st_direction"] = direction
        
        return df["st_direction"]
    
    df["str_supertrend"] = calc_supertrend(df)
    df["str_supertrend"] = df["str_supertrend"].astype(int).astype(str).replace({"-1":"🔴", "1":"🟢", "0":""})
    # supertrend = df["st_direction"].iloc[-1]


    #------------- HMA + Supertrend
    df['str_hullma_supertrend'] = np.where((df['sign_hullma'] == "🟢") & (df['str_supertrend'] == "🟢"), "🟢", 
                                  np.where((df['sign_hullma'] == "🔴") & (df['str_supertrend'] == "🔴"), "🔴", ""))


    # ----------- 19. Stochastic K/D crossover en zonas XTREMs
    """
    Tu sistema tiene Stochastic calculado pero comentado. Esta señal aprovecha el cruce K/D en zonas XTREMs.
    """
    df["sign_stochas"] = np.where((df["STOCH_K"] < 25) & (df["STOCH_D"] < 25) & (df["STOCH_K"].shift(1) < df["STOCH_D"].shift(1)) & (df["STOCH_K"] > df["STOCH_D"]), "🟢",
                         np.where((df["STOCH_K"] > 75) & (df["STOCH_D"] > 75) & (df["STOCH_K"].shift(1) > df["STOCH_D"].shift(1)) & (df["STOCH_K"] < df["STOCH_D"]), "🔴", ""))

    # stochastic = df["stochcross_sign"].iloc[-1]


    # ----------- 20. Chaikin Money Flow (CMF)
    # Mide la presión compradora/vendedora ponderada por volumen durante N períodos. Más sensible que el OBV a cambios intradía.
    CMF_PERIOD = 20
    CMF_UMBRAL = 0.05

    df["CLV"] = ((df["Close"] - df["Low"]) - (df["High"] - df["Close"])) / (df["High"] - df["Low"] + 1e-9)
    df["CMF"] = (df["CLV"] * df["Volume"]).rolling(CMF_PERIOD).sum() / df["Volume"].rolling(CMF_PERIOD).sum()

    df["sign_cmf"] = np.where((df["CMF"] > CMF_UMBRAL) & (df["CMF"] > df["CMF"].shift(1)), "🟢", 
                     np.where((df["CMF"] < -CMF_UMBRAL) & (df["CMF"] < df["CMF"].shift(1)), "🔴", ""))
    # cmf = df["CMF_sign"].iloc[-1]
    del df["CLV"], df["CMF"]



    # ----------- 21. Parabolic SAR
    # Excelente para gestión de stops dinámicos y detección de reversiones. Complementa al Supertrend — juntos forman una doble confirmación ATR-based.
    def calc_psar(df, iaf=0.02, maxaf=0.2):

        df = df.copy()

        length = len(df)
        high = df["High"].values
        low = df["Low"].values

        psar = np.zeros(length)

        bull = True
        af = iaf
        hp = high[0]
        lp = low[0]

        # Inicialización
        psar[0] = low[0]
        psar[1] = low[0]

        for i in range(2, length):

            if bull:
                psar[i] = psar[i-1] + af * (hp - psar[i-1])
                psar[i] = min(psar[i], low[i-1], low[i-2])

                if low[i] < psar[i]:
                    bull = False
                    psar[i] = hp
                    lp = low[i]
                    af = iaf
                else:
                    if high[i] > hp:
                        hp = high[i]
                        af = min(af + iaf, maxaf)

            else:
                psar[i] = psar[i-1] + af * (lp - psar[i-1])
                psar[i] = max(psar[i], high[i-1], high[i-2])

                if high[i] > psar[i]:
                    bull = True
                    psar[i] = lp
                    hp = high[i]
                    af = iaf
                else:
                    if low[i] < lp:
                        lp = low[i]
                        af = min(af + iaf, maxaf)

        df["PSAR"] = psar

        # Señal
        df["sign_psar"] = np.where(df["Close"] > df["PSAR"], "🟢", np.where(df["Close"] < df["PSAR"], "🔴", ""))

        return df["sign_psar"], df["PSAR"]
    
    df["sign_psar"], df["PSAR"] = calc_psar(df)
    # psar = df["PSAR"].iloc[-1]


    # # ----------- 22. Elder Ray Index
    # Mide la diferencia entre el precio y una EMA. El Bull Power es High - EMA13, Bear Power es Low - EMA13. 
    # Detecta si los compradores/vendedores están ganando fuerza dentro de la tendencia.

    EMA_ELDER = 13
    df["EMA13"]      = df["Close"].ewm(span=EMA_ELDER, adjust=False).mean()
    df["bull_power"] = df["High"] - df["EMA13"]
    df["bear_power"] = df["Low"]  - df["EMA13"]

    # Señal: tendencia alcista con bear_power cruzando hacia positivo (compresión terminando)
    df["sign_eri"] = np.where((df["bull_power"].shift(1) > 0) & (df["bear_power"].shift(1) < 0) & (df["bear_power"].shift(2) < df["bear_power"].shift(1)), "🟢",
                     np.where((df["bull_power"].shift(1) < 0) & (df["bear_power"].shift(1) > 0) & (df["bull_power"].shift(2) < df["bull_power"].shift(1)), "🟢", ""))
    # elder_ray = df["edr_sign"].iloc[-1]
    del df["EMA13"], df["bull_power"], df["bear_power"] 


    # ----------- 23. Williams %R
    # Similar al Stochastic pero más reactivo. Útil para detectar zonas de agotamiento rápido, especialmente en intradía.
    WILLR_PERIOD = 14
    highest_high = df["High"].rolling(WILLR_PERIOD).max()
    lowest_low   = df["Low"].rolling(WILLR_PERIOD).min()
    df["WillR"]  = -100 * (highest_high - df["Close"]) / (highest_high - lowest_low + 1e-9)

    # Saliendo de zonas XTREMs (más preciso que entrar en ellas)
    df["sign_willr"] = np.where((df["WillR"].shift(-2) < -80) & (df["WillR"].shift(-1) > -80), "🟢",   # saliendo de sobreventa
                        np.where((df["WillR"].shift(-2) > -20) & (df["WillR"].shift(-1) < -20), "🔴", "")) # saliendo de sobrecompra


    # ----------- 25. Donchian Channel Breakout
    """
    Captura rupturas de rango después de consolidación. Complementa perfectamente al Squeeze porque el Squeeze dice cuándo viene el movimiento y Donchian dice si ya rompió.
    Detecta ruptura de máximos/mínimos recientes (tendencia fuerte)
    """
    n = 20
    df["donchian_high"] = df["High"].rolling(n).max()
    df["donchian_low"]  = df["Low"].rolling(n).min()

    df["sign_donchian"] = np.where(df["Close"] > df["donchian_high"].shift(1), "🟢",
                          np.where(df["Close"] < df["donchian_low"].shift(1), "🔴", ""))
    # donchian = df["donchian_sign"].iloc[-1]


    # ----------- 26. Liquidity Sweep (caza de stops)
    #  Muy usado por smart money.
    # Lógica
    # Rompe máximo previo pero cierra por debajo → trampa → SELL
    # Rompe mínimo previo pero cierra por encima → BUY

    df["prev_high"] = df["High"].shift(1)
    df["prev_low"] = df["Low"].shift(1)

    df["sign_liquid"] = np.where((df["High"] > df["prev_high"]) & (df["Close"] < df["prev_high"]), "🔴", 
                        np.where((df["Low"] < df["prev_low"]) & (df["Close"] > df["prev_low"]), "🟢", ""))
    # liquidity = df["liquid_sign"].iloc[-1]
    del df["prev_high"], df["prev_low"]

    

    # ----------- 28. Divergencia MACD
    def compute_macd(df, fast=12, slow=26, signal=9):
        ema_fast = df["Close"].ewm(span=fast, adjust=False).mean()
        ema_slow = df["Close"].ewm(span=slow, adjust=False).mean()

        macd = ema_fast - ema_slow
        macd_signal = macd.ewm(span=signal, adjust=False).mean()
        hist = macd - macd_signal

        df = df.copy()
        df["MACD"] = macd
        df["MACD_signal"] = macd_signal
        df["MACD_hist"] = hist

        return df


    def find_pivots(series, left=3, right=3):
        pivots_high = []
        pivots_low = []

        for i in range(left, len(series) - right):
            window = series[i-left:i+right+1]

            if series.iloc[i] == window.max():
                pivots_high.append(i)

            if series.iloc[i] == window.min():
                pivots_low.append(i)

        return pivots_high, pivots_low


    def detect_macd_divergence(df, left=3, right=3):

        df = compute_macd(df)

        price = df["Close"]
        macd = df["MACD"]

        highs, lows = find_pivots(price, left, right)

        signals = pd.DataFrame(index=df.index)
        signals["bullish_div"] = False
        signals["bearish_div"] = False

        for i in range(1, len(lows)):
            prev = lows[i-1]
            curr = lows[i]
            confirm = curr + right          # vela donde el pivote ya es observable
            if confirm >= len(df):
                continue
            price_ll = price.iloc[curr] < price.iloc[prev]
            macd_hl = macd.iloc[curr] > macd.iloc[prev]
            if price_ll and macd_hl:
                signals.loc[df.index[confirm], "bullish_div"] = True

        for i in range(1, len(highs)):
            prev = highs[i-1]
            curr = highs[i]
            confirm = curr + right
            if confirm >= len(df):
                continue
            price_hh = price.iloc[curr] > price.iloc[prev]
            macd_lh = macd.iloc[curr] < macd.iloc[prev]
            if price_hh and macd_lh:
                signals.loc[df.index[confirm], "bearish_div"] = True

        return signals["bearish_div"], signals["bullish_div"]

    df["bearish_div"], df["bullish_div"] = detect_macd_divergence(df)

    df["tempsignal"] = np.where(df["bullish_div"], "🟢", np.where(df["bearish_div"], "🔴", ""))

    #----------- V1
    window = 1
    recent_signal = df["tempsignal"].replace("", np.nan).ffill(limit=window)
    df["str_macd_div"] = recent_signal.fillna("")
    df["str_macd_div1"] = df["str_macd_div"].where(df["str_macd_div"] != df["str_macd_div"].shift(1), "")

    # df["str_macd_div1"] = np.where(df["bullish_div"], "🟢",  np.where(df["bearish_div"], "🔴", ""))

    #----------- V2
    window = 3
    recent_signal = (df["tempsignal"].replace("", np.nan).ffill(limit=window))
    df["str_macd_div"] = recent_signal.fillna("")
    df["str_macd_div3"] = df["str_macd_div"].where(df["str_macd_div"] != df["str_macd_div"].shift(1), "")

    #----------- V3
    window = 5
    recent_signal = (df["tempsignal"].replace("", np.nan).ffill(limit=window))
    df["str_macd_div"] = recent_signal.fillna("")
    df["str_macd_div5"] = df["str_macd_div"].where(df["str_macd_div"] != df["str_macd_div"].shift(1), "")


    del df["tempsignal"], df["bullish_div"], df["bearish_div"], df["str_macd_div"]




    #---------------------------------- OTHER STRATEGIES --------------------------------#

    # C. PULLBACK PERFECTO
    buy_pullback = (
        df["trend"].isin(["▲ 2", "▲ 3", "▲ 4 XTREM", "▲ 5 SUPER XTREM"]) &
        (df["str_pullb_ema"] == "🟢") &
        (df["RSI"] < 30)
    )

    sell_pullback = (
        df["trend"].isin(["▼ 2", "▼ 3", "▼ 4 XTREM", "▼ 5 SUPER XTREM"]) &
        (df["str_pullb_ema"] == "🔴") &
        (df["RSI"] > 70)
    )
    # Se añade rolling
    buy_pullback = buy_pullback.rolling(1, min_periods=1).sum()
    sell_pullback = sell_pullback.rolling(1, min_periods=1).sum()

    df["strat_pullback"] = np.where(buy_pullback > sell_pullback, "🟢",
                            np.where(sell_pullback > buy_pullback, "🔴", ""))


    # # D. SQUEEZE
    # sign_numeric = df["squeeze_sign"].map({"🟢": 1, "🔴": -1}).fillna(0)
    # buy_count  = (sign_numeric == 1).rolling(3, min_periods=1).sum()
    # sell_count = (sign_numeric == -1).rolling(3, min_periods=1).sum()
    # df["squeeze_sign"] = np.where(buy_count > sell_count, "🟢", np.where(sell_count > buy_count, "🔴", ""))

    # buy_squeeze_combo = (
    #     (df["compress_sign"] == "✅") &
    #     (df["squeeze_sign"] == "🟢") &
    #     (df["momentum_sign"] == "🟢")
    # )

    # sell_squeeze_combo = (
    #     (df["compress_sign"] == "✅") &
    #     (df["squeeze_sign"] == "🔴") &
    #     (df["momentum_sign"] == "🔴")
    # )
    # Se añade rolling
    # buy_squeeze_combo = buy_squeeze_combo.rolling(3, min_periods=1).sum()
    # sell_squeeze_combo = sell_squeeze_combo.rolling(3, min_periods=1).sum()

    # # No se cumple nunca ninguna de las condiciones, si se cumplen si se aumenta rolling de squeeze
    # df["strat_squeeze"] = np.where(buy_squeeze_combo > sell_squeeze_combo, "🟢",
    #                         np.where(sell_squeeze_combo > buy_squeeze_combo, "🔴", ""))
    

    # E. ESTRUCTURA
    buy_structure = (
        (df["str_3ema"] == "🟢") &
        (df["sign_hullma"] == "🟢") &
        (df["sign_obv"] == "🟢") &
        (df["Close"] > df["EMA200"])
    )

    sell_structure = (
        (df["str_3ema"] == "🔴") &
        (df["sign_hullma"] == "🔴") &
        (df["sign_obv"] == "🔴") &
        (df["Close"] < df["EMA200"])
    )

    # Se añade rolling
    buy_structure = buy_structure.rolling(1, min_periods=1).sum()
    sell_structure = sell_structure.rolling(1, min_periods=1).sum()

    df["strat_struct"] = np.where(buy_structure > sell_structure, "🟢",
                            np.where(sell_structure > buy_structure, "🔴", ""))


    df["symbol"] = symbol
    return df # results


from joblib import Parallel, delayed

def main(SYMBOLS, INTERVAL, LOOKBACK_DAYS, DAY, HOUR, SIGNAL = "str_macd_div1", atr_sl = 2, multiplier_tp = 2):
    # print(f"Se analizan {len(SYMBOLS)} símbolos")
    # print(f"Intervalo: {INTERVAL}  |  Lookback: {LOOKBACK_DAYS} días")

    output = []
    chunk_size = 90
    for i in range(0, len(SYMBOLS), chunk_size):
        chunk = SYMBOLS[i:i + chunk_size]
        chunk_results = Parallel(n_jobs=-1)(
            delayed(analyze_symbol)(sym, INTERVAL, LOOKBACK_DAYS, DAY, HOUR)
            for sym in chunk)
        output.extend(chunk_results)

    # valid_rows = [row for row in output if row is not None and len(row) == 2]
    # results = pd.concat([row[0] for row in valid_rows if row[0] is not None], ignore_index=True)
    # df = pd.concat([row[1] for row in valid_rows if row[1] is not None], ignore_index=True)
      
    df = pd.concat(output)

    df = df.replace({False:"", True:"✅"})
    

    for signal in [SIGNAL]:
        try:
            mapping = {"🔴": -1, "": 0, "🟢": 1}
            df[signal] = df[signal].replace(mapping).fillna(0).astype(int)
        except:
            print("ERROR:", signal)


    df["signal"] = df[SIGNAL]
    df["sl_dist"] = atr_sl*df["ATR"]
    df["sl"] = np.where(df["signal"] == 1, df["Close"] - df["sl_dist"], 
                    np.where(df["signal"] == -1, df["Close"] + df["sl_dist"], np.nan))
    df["tp_dist"] = abs(df["Close"] - df["sl"]) * multiplier_tp
    df["tp"] = np.where(df["signal"] == 1, df["Close"] + df["tp_dist"], 
                    np.where(df["signal"] == -1, df["Close"] - df["tp_dist"], np.nan))

    for j in df.columns:
        try:
            df[j] = df[j].round(3)
        except:
            pass
        
    cols = ["symbol",  "signal", "Close", "sl", "tp", "bb_pos", "trend", "RSI", "ADX"]
    cols = cols + [c for c in df.columns if c not in cols]
    df=df[cols]
    df=df[df["signal"]!=0]

    return df

## Signals

In [ ]:
results =  main(SYMBOLS = symbs, INTERVAL = "1h", LOOKBACK_DAYS = 60, DAY=None, HOUR=None, SIGNAL = "str_macd_div1", atr_sl = 2, multiplier_tp = 2)
df = results.copy()

results = results.sort_values(["symbol", "date"]).drop_duplicates("symbol", keep = "last")   

print(results["str_macd_div1"].value_counts(dropna = False))

results 

In [ ]:
# Plots
for symb in results["symbol"]:
    df_plot = df[df["symbol"] == symb].tail(500)
    df_plot["date"] = pd.to_datetime(df_plot["date"])
    df_plot.index = df_plot["date"]
    signal = results[results["symbol"] == symb]["signal"].iloc[-1]

    row = df[df["symbol"] == symb].iloc[-1]
    buy_signals = pd.DataFrame(columns = ["price", "index"])
    sell_signals = pd.DataFrame(columns = ["price", "index"])
    if signal == 1:
        buy_signals = pd.DataFrame({"price":[row["Close"]], "index":[row["date"]]})
    if signal == -1:
        sell_signals = pd.DataFrame({"price":[row["Close"]], "index":[row["date"]]})

    df_plot = df_plot.copy().dropna(subset=["Close"])
    
    import warnings
    warnings.filterwarnings("ignore")

    import mplfinance as mpf
    import matplotlib.pyplot as plt

    # =========================
    # PREPARAR DATA
    # =========================
    df_candle = df_plot.copy()
    df_candle.index = pd.to_datetime(df_candle.index)

    df_candle = df_candle[["Open","High","Low","Close","EMA20","EMA50","EMA200", "RSI"]].dropna()

    # =========================
    # SEÑALES
    # =========================
    buy_plot = np.full(len(df_candle), np.nan)
    sell_plot = np.full(len(df_candle), np.nan)

    index_map = {k: i for i, k in enumerate(df_candle.index)}

    for _, row in buy_signals.iterrows():
        if row["index"] in index_map:
            buy_plot[index_map[row["index"]]] = row["price"] #* 0.85

    for _, row in sell_signals.iterrows():
        if row["index"] in index_map:
            sell_plot[index_map[row["index"]]] = row["price"] #* 1.15

    # =========================
    # ESTILO OSCURO PRO
    # =========================
    mc = mpf.make_marketcolors(
        up='lime',
        down='red',
        edge='inherit',  
        wick='inherit',
        volume='gray'
    )

    style = mpf.make_mpf_style(
        base_mpf_style='nightclouds',
        marketcolors=mc,
        facecolor='black',
        figcolor='black',
        gridstyle=''
    )

    # =========================
    # EMAs
    # =========================
    ema20 = mpf.make_addplot(df_candle["EMA20"], color='yellow', width=1)
    ema50 = mpf.make_addplot(df_candle["EMA50"], color='orange', width=1)
    ema200 = mpf.make_addplot(df_candle["EMA200"], color='magenta', width=1)

    # RSI (panel inferior)
    # =========================
    rsi_plot = mpf.make_addplot(
        df_candle["RSI"],
        panel=1,
        color='white',
        width=1
    )

    rsi_70 = mpf.make_addplot(
        [70]*len(df_candle),
        panel=1,
        color='red',
        linestyle='dashed',
        width=0.8
    )

    rsi_30 = mpf.make_addplot(
        [30]*len(df_candle),
        panel=1,
        color='lime',
        linestyle='dashed',
        width=0.8
    )

    rsi_50 = mpf.make_addplot(
        [50]*len(df_candle),
        panel=1,
        color='white',
        linestyle='dashed',
        width=0.8
    )

    # =========================
    # SEÑALES
    # =========================
    apds = [ema20, ema50, ema200, rsi_plot, rsi_70, rsi_30, rsi_50]

    if np.any(~np.isnan(buy_plot)):
        apds.append(mpf.make_addplot(buy_plot, type='scatter', marker='^', markersize=90, color='lime'))

    if np.any(~np.isnan(sell_plot)):
        apds.append(mpf.make_addplot(sell_plot, type='scatter', marker='v', markersize=90, color='red'))


    fig, axlist = mpf.plot(
        df_candle,
        type='candle',
        style=style,
        addplot=apds,
        figsize=(18,5.8),
        title=symb,
        volume=False,
        datetime_format='%Y-%m-%d',
        returnfig=True,
        panel_ratios=(2.7, 0.39) 
    )

    for ax in axlist:
        ax.tick_params(axis='x', which='both', bottom=False, top=False, labelbottom=False)
    for ax in axlist:
        ax.tick_params(axis='y', labelsize=8)
    for label in axlist[-1].get_yticklabels():
        label.set_rotation(0)
        label.set_horizontalalignment('center')
        label.set_fontsize(8)

    plt.show()

In [ ]:
qwe

## MT5

In [ ]:
ajustar para que cierre posicion si hay reversion de la señal

In [ ]:
import time
from datetime import datetime
import warnings
import math
import MetaTrader5 as mt5
warnings.filterwarnings("ignore")

capital = 1000
risk_pct = 0.01  
k = 0

while True:
    k +=1
    now = datetime.now()
    # print(f"[ITER {k}]", now)
    
    def error_sound():
        import winsound
        for j in range(8):
            winsound.Beep(800, 200)

    hour = now.hour
    minute = now.minute
    second = now.second

    if minute == 2:
        print(now)
 
        results =  main(SYMBOLS = symbs, INTERVAL = "1h", LOOKBACK_DAYS = 60, DAY=None, HOUR=None, SIGNAL = "str_macd_div1", atr_sl = 2, multiplier_tp = 2)
        df = results.copy()

        results = results.sort_values(["symbol", "date"]).drop_duplicates("symbol", keep = "last")        

        def hay_posicion(symbol):
            posiciones = mt5.positions_get(symbol=symbol)
            return posiciones is not None and len(posiciones) > 0

        def calcular_lotaje(
                            capital,
                            risk_pct,
                            entry_price,
                            stop_loss,
                            contract_size,
                            volume_step,
                            volume_min,
                            volume_max
                            ):
            
            risk_money = capital * risk_pct               # dinero máximo a perder
            sl_distance = abs(entry_price - stop_loss)    # distancia al stop
            if sl_distance == 0:                          # evitar división por cero
                return None
            lot = risk_money / (sl_distance * contract_size)    # lote teórico 
            lot = math.floor(lot / volume_step) * volume_step   # ajustar al step permitido
            
            # respetar límites broker
            lot = max(volume_min, lot)
            lot = min(volume_max, lot)
            return round(lot, 4)
        

        if not mt5.initialize():
            print("Error al conectar:", mt5.last_error())
            error_sound()
            quit()

        if len(results) > 0:
            # print(now)
            print("Signals:", len(results)) 
            import winsound
            winsound.Beep(500, 4000)

        for _, row in results.iterrows():
            try:
                symbol = row["symbol"].replace("-","")
                signal = row["signal"]
                sl = row["sl"]
                tp = row["tp"]

                print(f"\n------------------------ {symbol} ------------------------")

                mt5.symbol_select(symbol, True)
                tick = mt5.symbol_info_tick(symbol)
                info = mt5.symbol_info(symbol)

                if tick is None or info is None:
                    print("❌ Error obteniendo información")
                    error_sound()
                    continue

                if hay_posicion(symbol):
                    print("⚠️ Ya existe posición abierta")
                    continue

                digits = info.digits
                contract_size = info.trade_contract_size

                volume_min = info.volume_min
                volume_max = info.volume_max
                volume_step = info.volume_step

                stop_level = info.trade_stops_level

                if signal == 1:
                    order_type = mt5.ORDER_TYPE_BUY
                    price = tick.ask

                    if sl >= price:
                        print("❌ SL inválido para BUY")
                        error_sound()
                        continue
                    if tp <= price:
                        print("❌ TP inválido para BUY")
                        error_sound()
                        continue

                elif signal == -1:
                    order_type = mt5.ORDER_TYPE_SELL
                    price = tick.bid

                    if sl <= price:
                        print("❌ SL inválido para SELL")
                        error_sound()
                        continue

                    if tp >= price:
                        print("❌ TP inválido para SELL")
                        error_sound()
                        continue
                else:
                    print("❌ Señal inválida")
                    error_sound()
                    continue

                price = round(price, digits)
                sl = round(sl, digits)
                tp = round(tp, digits)

                lot = calcular_lotaje(
                                        capital=capital,
                                        risk_pct=risk_pct,
                                        entry_price=price,
                                        stop_loss=sl,
                                        contract_size=contract_size,
                                        volume_step=volume_step,
                                        volume_min=volume_min,
                                        volume_max=volume_max
                                    )

                if lot is None:
                    print("❌ Error calculando lote")
                    error_sound()
                    continue

                sl_distance = abs(price - sl)
                real_risk = sl_distance * contract_size * lot
                position_money = lot * contract_size * price

                print(f"Signal:            {signal}")
                print(f"Price:             {price}")
                print(f"SL:                {sl}")
                print(f"TP:                {tp}")

                print("-" * 50)

                print(f"Capital:           ${capital:.2f}")
                print(f"Risk target:       ${capital * risk_pct:.2f}")
                print(f"Real risk:         ${real_risk:.2f}")

                print("-" * 50)

                print(f"Lot:               {lot}")
                print(f"Min lot:           {volume_min}")
                print(f"Contract size:     {contract_size}")

                print("-" * 50)

                print(f"Position value:    ${position_money:.2f}")


                tick = mt5.symbol_info_tick(symbol)
                request = {
                    "action": mt5.TRADE_ACTION_DEAL,
                    "symbol": symbol,
                    "volume": lot,
                    "type": order_type,
                    "price": tick.ask,
                    "sl": sl,
                    "tp": tp,
                    "deviation": 20,
                    "comment": "bot_python",
                    "type_time": mt5.ORDER_TIME_GTC,
                    "type_filling": mt5.ORDER_FILLING_IOC,
                }

                check = mt5.order_check(request)

                # print("\nOrder Check:")
                # print(check)

                result = mt5.order_send(request)

                # print("\nOrder Send:")
                print(result)



                if result.retcode == mt5.TRADE_RETCODE_DONE:
                    print("\n✅ Trade ejecutado")
                    state = "✅ Trade ejecutado"
                    import winsound
                    winsound.Beep(1000, 1000)
                    winsound.Beep(1200, 1000)
                    winsound.Beep(800, 1000)
                    time.sleep(5)
                else:
                    print(f"\n❌ Error: {result.comment}")
                    error_sound()

                print("=" * 100)

                
                # Save log
                now = datetime.now()
                now = str(now)

                log = {
                    "date": now,
                    "symbol": symbol,
                    "signal": signal,
                    "state": state,
                    "price": price,
                    "sl": sl,
                    "tp": tp,
                    "capital": capital,
                    "real_risk": real_risk,
                    "lot": lot,
                    "position_money": position_money,
                    "risk_target": capital * risk_pct,
                    "sl_distance": sl_distance,
                    "volume_min": volume_min,
                    "contract_size": contract_size,
                    "request": request
                }

                log = pd.DataFrame([log])

                log.to_csv(f"C:/Users/PC/Desktop/log trades/{now}.csv", sep="|", decimal=".", index=False)


            except Exception as e:
                print(f"\n❌ ERROR GENERAL: {e}")
                error_sound()
                print("=" * 100)

        time.sleep(2)

    time.sleep(30) 
    # print("="*180)
    # print("\n")


In [ ]:
qwe

## Simulando trades MT5

Desde tabla

In [ ]:
INTERVAL        = "1h"          # Ventana velas
LOOKBACK_DAYS   = 60            # 1h: 60 | 4h: 240 | 1d: 1440
results =  main(SYMBOLS = symbs, INTERVAL = INTERVAL, LOOKBACK_DAYS = LOOKBACK_DAYS, DAY = None, HOUR = None)
df = results.copy()
results = results.replace({False:"", True:"✅"})
SIGNAL = "str_macd_div1"

for signal in ["str_macd_div1"]:
    try:
        mapping = {"🔴": -1, "": 0, "🟢": 1}
        results[signal] = results[signal].replace(mapping).fillna(0).astype(int)
    except:
        print("ERROR:", signal)
        
# results["str_macd_div1"] = results["str_macd_div1"].rolling(2).sum()
# results["str_macd_div1"] = np.where(results["str_macd_div1"] >= 1, 1, np.where(results["str_macd_div1"] <= -1, -1, 0))

results = results.sort_values(["symbol", "date"])#.drop_duplicates("symbol", keep = "last")
results
atr_sl                = 2        # Value for SL. Scalping: 0.8-1.5 | Intradia: 1.5-2.5 | Swing: 2-3.5 | Largo: 3-5
multiplier_tp         = 2        # Value for TP -> atr_tp = atr_sl x multiplier_tp. 

results["sl_dist"] = atr_sl*results["ATR"]
results["sl"] = np.where(results["str_macd_div1"] == 1, results["Close"] - results["sl_dist"], 
                np.where(results["str_macd_div1"] == -1, results["Close"] + results["sl_dist"], np.nan))
results["tp_dist"] = abs(results["Close"] - results["sl"]) * multiplier_tp
results["tp"] = np.where(results["str_macd_div1"] == 1, results["Close"] + results["tp_dist"], 
                np.where(results["str_macd_div1"] == -1, results["Close"] - results["tp_dist"], np.nan))
results["signal"] = results["str_macd_div1"]

for j in results.columns:
    try:
        results[j] = results[j].round(3)
    except:
        pass

cols = ["symbol",  "signal", "Close", "sl", "tp", "bb_pos", "trend", "RSI", "ADX"]
cols = cols + [c for c in results.columns if c not in cols]
results=results[cols]
# results=results[results["signal"]!=0]
# results=results[results.symbol.isin(["BTC-USD","ETH-USD"])]
print("Total signals:", len(results)) 

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_colwidth', 200)

print(results["str_macd_div1"].value_counts(dropna = False))

results 

In [ ]:
results["hour"] = results["date"].dt.strftime("%H").astype(int)
results["day"] = results["date"].dt.strftime("%Y-%m-%d")

In [ ]:
date = "2026-06-09"
symb = "BTC-USD"
hour_min = 8
hour_max = 20
ax = results[results.symbol.isin([symb])]
ax = ax[(ax["day"] == date) & (ax["hour"] >= hour_min) & (ax["hour"] <= hour_max)]
ax

In [ ]:
print(sorted(results["day"].unique(), reverse = True))

In [ ]:
symbs_exe = results["symbol"].unique()
len(symbs_exe)

In [ ]:
days = ['2026-06-09', '2026-06-08', '2026-06-07', '2026-06-06', '2026-06-05', '2026-06-04', '2026-06-03', '2026-06-02', '2026-06-01', '2026-05-31', '2026-05-30', '2026-05-29', 
        '2026-05-28', '2026-05-27', '2026-05-26', '2026-05-25', '2026-05-24', '2026-05-23', '2026-05-22', '2026-05-21', '2026-05-20', '2026-05-19', '2026-05-18', '2026-05-17']
print(len(days))

for date in days:
    count = 0
    # for symb in ["BTC-USD"]: # symbs_exe:
    for symb in symbs_exe:
        # print(symb)
#     symb = "ETH-USD"
        ax = results[results.symbol.isin([symb])].copy()
        ax = ax[(ax["day"] == date) & (ax["hour"] >= hour_min) & (ax["hour"] <= hour_max)]
        ax = ax[ax["signal"]!= 0]
        count += len(ax)

    print(f"{date}: {count}")

In [ ]:
qwe

Desde ejecucion main

In [ ]:
days = ['2026-06-09', '2026-06-08', '2026-06-07', '2026-06-06', '2026-06-05', '2026-06-04', '2026-06-03', '2026-06-02', '2026-06-01', '2026-05-31', '2026-05-30', '2026-05-29', 
        '2026-05-28', '2026-05-27', '2026-05-26', '2026-05-25', '2026-05-24', '2026-05-23', '2026-05-22', '2026-05-21', '2026-05-20', '2026-05-19', '2026-05-18', '2026-05-17']
print(len(days))

hours = list(range(8, 19))
# hours = [16]
print(hours)

In [ ]:
import warnings
warnings.filterwarnings("ignore")
for date in days[:5]:    # Fecha en la que que se quiere obtener la señal (se toma todos los registros con fecha < date y se queda con el ultimo (con hora))
    print(f"- {date}")
    count = 0
    for hour in hours:      # Hora en la que se quiere obtener la señal
        # date = "2026-06-03"
        # hour = 14       
        # INTERVAL        = "1h"          # Ventana velas
        LOOKBACK_DAYS   = 60            # 1h: 60 | 4h: 240 | 1d: 1440
        results =  main(SYMBOLS = symbs, INTERVAL = INTERVAL, LOOKBACK_DAYS = LOOKBACK_DAYS, DAY = date, HOUR = hour)

        df = results.copy()
        results = results.replace({False:"", True:"✅"})
        SIGNAL = "str_macd_div1"

        for signal in ["str_macd_div1"]:
            try:
                mapping = {"🔴": -1, "": 0, "🟢": 1}
                results[signal] = results[signal].replace(mapping).fillna(0).astype(int)
            except:
                print("ERROR:", signal)
                
        # results["str_macd_div1"] = results["str_macd_div1"].rolling(2).sum()
        # results["str_macd_div1"] = np.where(results["str_macd_div1"] >= 1, 1, np.where(results["str_macd_div1"] <= -1, -1, 0))

        results = results.sort_values(["symbol", "date"]).drop_duplicates("symbol", keep = "last")

        for signal in ["str_macd_div1"]:
            try:
                mapping = {"🔴": -1, "": 0, "🟢": 1}
                results[signal] = results[signal].replace(mapping).fillna(0).astype(int)
            except:
                print("ERROR:", signal)
                
        results
        atr_sl                = 2        # Value for SL. Scalping: 0.8-1.5 | Intradia: 1.5-2.5 | Swing: 2-3.5 | Largo: 3-5
        multiplier_tp         = 2        # Value for TP -> atr_tp = atr_sl x multiplier_tp. 

        results["sl_dist"] = atr_sl*results["ATR"]
        results["sl"] = np.where(results["str_macd_div1"] == 1, results["Close"] - results["sl_dist"], 
                        np.where(results["str_macd_div1"] == -1, results["Close"] + results["sl_dist"], np.nan))
        results["tp_dist"] = abs(results["Close"] - results["sl"]) * multiplier_tp
        results["tp"] = np.where(results["str_macd_div1"] == 1, results["Close"] + results["tp_dist"], 
                        np.where(results["str_macd_div1"] == -1, results["Close"] - results["tp_dist"], np.nan))
        results["signal"] = results["str_macd_div1"]

        for j in results.columns:
            try:
                results[j] = results[j].round(3)
            except:
                pass

        cols = ["symbol",  "signal", "Close", "sl", "tp", "bb_pos", "trend", "RSI", "ADX"]
        cols = cols + [c for c in results.columns if c not in cols]
        results=results[cols]
        results=results[results["signal"]!=0]
        print(f"{hour}: {len(results)} {results["symbol"].tolist() if len(results) > 0 else ''}") 

        count += len(results)
    print("Total trades:", count)

    pd.set_option('display.max_colwidth', 200)
    print(" ")

In [ ]:
ax = results[results.symbol.isin(["FIL-USD"])]
ax = ax[ax["day"] == "2026-06-09"]
ax

In [ ]:
qwe

poner condicion en position_money < x

In [ ]:
qwe

In [ ]:
# Consumir historial de trades
date = "2026-04-15"
import MetaTrader5 as mt5
from datetime import datetime
import matplotlib.pyplot as plt
import pandas as pd, numpy as np

# =========================================================
# 1. CONEXIÓN
# =========================================================
if not mt5.initialize():
    raise RuntimeError(f"Error al conectar con MT5: {mt5.last_error()}")

# =========================================================
# 2. DESCARGAR HISTORIAL
# =========================================================
date_from = datetime(2024, 1, 1)
date_to = datetime.now()

deals = mt5.history_deals_get(date_from, date_to)

if deals is None or len(deals) == 0:
    mt5.shutdown()
    raise ValueError("No hay datos de trades en ese rango de fechas")

# =========================================================
# 3. CONVERTIR A DATAFRAME
# =========================================================
df = pd.DataFrame(list(deals), columns=deals[0]._asdict().keys())

# tiempo
df["time"] = pd.to_datetime(df["time"], unit="s")

# solo BUY / SELL
df = df[df["type"].isin([0, 1])]

# mapear tipo
df["type"] = df["type"].map({0: "buy", 1: "sell"})

# =========================================================
# 4. RECONSTRUIR TRADES (ENTRY + EXIT)
# =========================================================
trades = []

for position_id, group in df.groupby("position_id"):

    group = group.sort_values("time")

    if len(group) < 2:
        continue

    entry = group.iloc[0]
    exit_ = group.iloc[-1]

    trades.append({
        "position_id": position_id,
        "symbol": entry["symbol"],
        "type": entry["type"],
        "entry_time": entry["time"],
        "exit_time": exit_["time"],
        "entry_price": entry["price"],
        "exit_price": exit_["price"],
        "volume": entry["volume"],
        "profit": group["profit"].sum(),
        "swap": group["swap"].sum(),
        "comment": group["comment"].sum(), # Probar si es entry
        "commission": group["commission"].sum()
    })

trades_df = pd.DataFrame(trades)

# =========================================================
# 5. CALCULAR RETURNS
# =========================================================
trades_df["return"] = (
    (trades_df["exit_price"] - trades_df["entry_price"]) / trades_df["entry_price"]
)

trades_df.loc[trades_df["type"] == "sell", "return"] *= -1

trades_df["result"] = np.where(trades_df["profit"] > 0, "win", "loss")
trades_df["entry_time"] = pd.to_datetime(trades_df["entry_time"])
trades_df = trades_df[trades_df["entry_time"] >= date]

# =========================================================
# 6. MÉTRICAS
# =========================================================
winrate = (trades_df["result"] == "win").mean()
avg_return = trades_df["return"].mean()
total_profit = trades_df["profit"].sum()

wins = trades_df[trades_df["return"] > 0]["return"]
losses = trades_df[trades_df["return"] <= 0]["return"]

profit_factor = wins.sum() / abs(losses.sum()) if len(losses) > 0 else np.nan

expectancy = (
    winrate * wins.mean() + (1 - winrate) * losses.mean()
    if len(wins) > 0 and len(losses) > 0 else np.nan
)

print("\n========== MÉTRICAS ==========")
print("Trades:", len(trades_df))
print("Winrate:", round(winrate, 3))
print("Avg Return:", round(avg_return, 5))
print("Profit Factor:", round(profit_factor, 3))
print("Expectancy:", round(expectancy, 5))
print("Total Profit:", round(total_profit, 2))

# =========================================================
# 7. EQUITY CURVE
# =========================================================
trades_df = trades_df.sort_values("exit_time")

trades_df["equity"] = trades_df["profit"].cumsum()

plt.figure(figsize=(12,5))
plt.plot(trades_df["exit_time"], trades_df["equity"])
plt.title("Equity Curve (MT5 Real Trades)")
plt.grid()
plt.show()

# =========================================================
# 8. DRAWDOWN Y CALMAR
# =========================================================
equity = trades_df["equity"]

peak = equity.cummax()
drawdown = equity - peak
drawdown_pct = drawdown / peak

max_dd = drawdown_pct.min()

total_return = equity.iloc[-1] / abs(equity.iloc[0] if equity.iloc[0] != 0 else 1)
calmar = total_return / abs(max_dd) if max_dd != 0 else np.nan

print("\n========== RIESGO ==========")
print("Max Drawdown:", round(max_dd, 3))
print("Calmar:", round(calmar, 3))

# =========================================================
# 9. EXPORTAR (OPCIONAL)
# =========================================================
# trades_df.to_csv("mt5_trades.csv", index=False)

# cerrar conexión
mt5.shutdown()

In [ ]:
trades_df

In [ ]:
trades_df["comment"].value_counts()

In [ ]:
group

In [ ]:
entry

In [ ]:
# ayudame haciendo un algoritmo en python que me indique cuando colocar una pocision de venta y compra de FOREX, con estos simbolos: "BTC-USD", "ETH-USD", "XRP-USD", "LTC-USD", "ADA-USD", . Necesito que uses yahoo finance y operar a una ventana de 4 horas por vela. Necesito que cada vez que corra el algoritmo me indique por cada simbolo si debo comprar, vender, o esperar, crea las reglas teniendo en cuenta los indicadores tecnicos que necesites. Busca la mejor estrategia posible. Necesito que definas automaticamente el take profit y stop loss. Ademas pon un parametro del minimo de reglas que se cumplan para decidir o no colocar la posicion